# Chapter 15: Configurations, Theorems, and Bracket Expressions

**Source span read.** Perspectives on Projective Geometry, Chapter 15, sections 15.1-15.6, printed pp. 269-294 / PDF pp. 291-316.

**Chapter goal.** Build a computational proof machine for the chapter: incidence theorems become determinant equations, determinant equations become cancellable bracket products, and the same cancellation idea reappears in cross-ratio chains and glued Ceva/Menelaus configurations.

The notebook is standalone. It uses the source for mathematical orientation only; all prose, code, diagrams, checks, and artifacts here are original to this notebook.


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Perspectives-on-Projective-Geometry book root.")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

CHAPTER_SLUG = "chapter-15-configurations-theorems-and-bracket-expressions"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / CHAPTER_SLUG
for subdir in ("figures", "html", "tables", "checks"):
    (ARTIFACT_ROOT / subdir).mkdir(parents=True, exist_ok=True)

NOTEBOOK_ROOT = Path.cwd().resolve()
ARTIFACT_ROOT.relative_to(BOOK_ROOT)


## Computational Translation Guide

The chapter has one repeated translation pattern.

| Book object | Computational representation | Why it is useful |
| --- | --- | --- |
| Point in the projective plane | Homogeneous vector `(x, y, w)` | Joins, meets, and projective maps become cross products or matrix products. |
| Line through two points | Cross product `p x q` | Incidence is the scalar test `p dot line = 0`. |
| Bracket `[A B C]` | Determinant of three homogeneous point columns | Collinearity is a zero determinant; signs record orientation. |
| Binomial proof row | Equality of two products of brackets | Multiplying rows turns a proof into a cancellation ledger. |
| Cross-ratio chain | Product of 2 by 2 projective maps on a line | Each perspectivity is a Mobius map, so the cross-ratio is transported. |
| Ceva/Menelaus expression | Product of three oriented ratios | The value `+1` means concurrence; the value `-1` means a transversal. |
| Glued configurations | Oriented face/edge cancellation table | Interior edge factors cancel, leaving a boundary or final-face condition. |

**Library routing.** Matplotlib is used for durable 2D incidence diagrams; NetworkX for proof and cancellation dependency graphs; Plotly for the inspectable cross-ratio chain; SymPy for exact bracket identities; Pandas/CSV for proof ledgers. This is a projective and symbolic chapter, so mesh, GIS, and surface libraries would add machinery without exposing the core invariants.


In [ ]:
from collections import Counter
from itertools import combinations
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp

from utils.artifacts import (
    artifact_path,
    assert_artifacts,
    book_relative,
    display_artifact,
    image_stats,
    save_json,
    save_table,
)
from utils.projective import affine, bracket, cross_ratio, hpoint, join, meet

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
})

ARTIFACTS = []
TABLES = []
CHECKS = {}

COLORS = {
    "blue": "#2f6fbb",
    "orange": "#d77f2f",
    "green": "#2e8f5b",
    "red": "#c23b3b",
    "purple": "#7a5195",
    "gray": "#555555",
}


def save_fig(fig, filename, *, dpi=180):
    path = artifact_path(ARTIFACT_ROOT, "figures", filename)
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    ARTIFACTS.append(path)
    return path


def save_csv(rows, filename):
    path = save_table(rows, ARTIFACT_ROOT, "tables", filename)
    TABLES.append(path)
    return path


def hp(point):
    return hpoint(float(point[0]), float(point[1]))


def point2(point):
    return np.asarray(affine(point), dtype=float)


def line_box_points(line, xlim, ylim, tol=1e-10):
    a, b, c = [float(v) for v in line]
    pts = []
    for x in xlim:
        if abs(b) > tol:
            y = (-a * x - c) / b
            if ylim[0] - 1e-8 <= y <= ylim[1] + 1e-8:
                pts.append(np.array([x, y]))
    for y in ylim:
        if abs(a) > tol:
            x = (-b * y - c) / a
            if xlim[0] - 1e-8 <= x <= xlim[1] + 1e-8:
                pts.append(np.array([x, y]))
    unique = []
    for p in pts:
        if not any(np.linalg.norm(p - q) < 1e-7 for q in unique):
            unique.append(p)
    if len(unique) < 2:
        return None
    return max(combinations(unique, 2), key=lambda pair: np.linalg.norm(pair[0] - pair[1]))


def draw_projective_line(ax, line, xlim, ylim, **kwargs):
    pair = line_box_points(line, xlim, ylim)
    if pair is None:
        return
    p, q = pair
    ax.plot([p[0], q[0]], [p[1], q[1]], **kwargs)


def draw_points(ax, points, color, offset=(0.04, 0.04), size=42):
    for label, point in points.items():
        arr = np.asarray(point)
        xy = point2(point) if arr.shape == (3,) else np.asarray(point, dtype=float)
        ax.scatter([xy[0]], [xy[1]], s=size, color=color, zorder=4, edgecolor="white", linewidth=0.8)
        ax.text(xy[0] + offset[0], xy[1] + offset[1], label, color=color, weight="bold")


def oriented_ratio(a, x, b):
    a = np.asarray(a, dtype=float)
    x = np.asarray(x, dtype=float)
    b = np.asarray(b, dtype=float)
    v = b - a
    t = float(np.dot(x - a, v) / np.dot(v, v))
    return t / (1.0 - t)


def intersect_affine(p1, p2, p3, p4):
    return point2(meet(join(hp(p1), hp(p2)), join(hp(p3), hp(p4))))


def bracket_product(P, A, B, C, X, Y, Z):
    return (
        bracket(P, A, X) / bracket(P, X, B)
        * bracket(P, B, Y) / bracket(P, Y, C)
        * bracket(P, C, Z) / bracket(P, Z, A)
    )


def rel(path):
    return book_relative(path)


## Visualization-Planner Pass

A useful route through this chapter has to make the proof state visible. The storyboard below is implemented directly in the notebook and saved as a check artifact.


In [ ]:
STORYBOARD = {
    "chapter goal": "Use explicit projective constructions and bracket/ratio ledgers to see why Chapter 15 theorems close.",
    "source span read": "Chapter 15, printed pp. 269-294 / PDF pp. 291-316.",
    "concept inventory": [
        "Desargues perspective triangles and the axis of corresponding-side intersections",
        "binomial bracket proof rows from Grassmann-Pluecker relations",
        "chains and cycles of cross-ratios under perspectivities",
        "Ceva and Menelaus as projective ratio products",
        "glued Ceva/Menelaus configurations and 2-cycle cancellation",
        "symbolic and numeric theorem checks for incidence, ratios, and brackets",
    ],
    "library routing table": [
        {"concept": "incidence diagrams", "representation": "labeled 2D homogeneous-coordinate plots", "library": "Matplotlib", "why": "static projective configurations need precise labels and equal aspect"},
        {"concept": "proof dependencies", "representation": "directed proof graph and incidence graph", "library": "NetworkX", "why": "the chapter emphasizes combinatorial proof structure"},
        {"concept": "cross-ratio chains", "representation": "interactive line-by-line transport", "library": "Plotly", "why": "hover and pan make the transported values inspectable"},
        {"concept": "bracket identities", "representation": "exact determinants and ledgers", "library": "SymPy and Pandas", "why": "the core checks are algebraic cancellation checks"},
    ],
    "visual sequence": [
        {"artifact": "figures/proof-machines-route-map.png", "inspection target": "which invariant powers each section", "validation": "all planned concepts have an artifact"},
        {"artifact": "figures/desargues-perspective-axis.png", "inspection target": "X, Y, Z land on one axis", "validation": "line incidence residuals"},
        {"artifact": "figures/desargues-10-3-incidence-graph.png", "inspection target": "each Desargues point/line has degree 3", "validation": "bipartite degree counts"},
        {"artifact": "figures/binomial-cancellation-ledger.png", "inspection target": "which bracket factors survive multiplication", "validation": "canonical bracket exponent counts"},
        {"artifact": "figures/cross-ratio-chain-transport.png", "inspection target": "cross-ratio value stays constant along the chain", "validation": "maximum cross-ratio drift"},
        {"artifact": "html/cross-ratio-cycle-lab.html", "inspection target": "hover over transported points and compare coordinates", "validation": "cycle-closing residual"},
        {"artifact": "figures/ceva-menelaus-ratio-products.png", "inspection target": "+1 versus -1 ratio products", "validation": "oriented ratio products"},
        {"artifact": "figures/glued-ceva-cancellation-disk.png", "inspection target": "the shared edge quotient cancels", "validation": "oriented edge exponents"},
    ],
    "computational checks": [
        "Desargues axis residual < 1e-9",
        "cross-ratio chain residual < 1e-9",
        "Ceva product = +1 and Menelaus product = -1 within tolerance",
        "binomial cancellation leaves the Grassmann-Pluecker conclusion factors",
        "tetrahedron face-cycle exponents cancel to zero",
        "all generated artifacts exist and have nonzero size",
    ],
    "risks": [
        "Degenerate point choices can make a denominator vanish, so code records nonzero denominator checks.",
        "The bracket proof table is algebraic data, not copied textbook prose or page imagery.",
    ],
}

storyboard_path = save_json(STORYBOARD, ARTIFACT_ROOT, "checks", "storyboard.json")

route = nx.DiGraph()
route_nodes = [
    ("Desargues", "incidence"),
    ("Binomial rows", "brackets"),
    ("Cross-ratio chain", "projectivity"),
    ("Ceva/Menelaus", "ratio products"),
    ("Glued cycles", "edge cancellation"),
    ("Sanity checks", "numeric + exact"),
]
for node, tag in route_nodes:
    route.add_node(node, tag=tag)
route_edges = [
    ("Desargues", "Binomial rows", "encode collinearity"),
    ("Binomial rows", "Sanity checks", "cancel brackets"),
    ("Cross-ratio chain", "Ceva/Menelaus", "ratios on lines"),
    ("Ceva/Menelaus", "Glued cycles", "multiply faces"),
    ("Glued cycles", "Sanity checks", "boundary/final face"),
]
route.add_edges_from((u, v) for u, v, _ in route_edges)
pos = {
    "Desargues": (0.0, 1.0),
    "Binomial rows": (1.6, 1.0),
    "Cross-ratio chain": (0.0, 0.0),
    "Ceva/Menelaus": (1.6, 0.0),
    "Glued cycles": (3.2, 0.0),
    "Sanity checks": (3.2, 1.0),
}
fig, ax = plt.subplots(figsize=(10.5, 4.1))
node_colors = [COLORS["blue"], COLORS["purple"], COLORS["green"], COLORS["orange"], COLORS["red"], COLORS["gray"]]
nx.draw_networkx_nodes(route, pos, ax=ax, node_size=2500, node_color=node_colors, edgecolors="white", linewidths=1.5)
nx.draw_networkx_labels(route, pos, ax=ax, font_size=10, font_weight="bold", font_color="white")
nx.draw_networkx_edges(route, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=2.0, edge_color="#38404a")
edge_labels = {(u, v): label for u, v, label in route_edges}
nx.draw_networkx_edge_labels(route, pos, edge_labels=edge_labels, ax=ax, font_size=8, label_pos=0.55, bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "#dddddd"})
for node, tag in route_nodes:
    x, y = pos[node]
    ax.text(x, y - 0.27, tag, ha="center", va="center", fontsize=8, color="#222222")
ax.set_title("Chapter 15 proof machines: incidence, brackets, ratios, cycles", weight="bold")
ax.set_axis_off()
route_path = save_fig(fig, "proof-machines-route-map.png")
CHECKS["storyboard_visual_count"] = len(STORYBOARD["visual sequence"])
display_artifact(route_path, width=780)


## Desargues: a Perspective Pair of Triangles

Desargues's theorem starts as a visible incidence claim. Two triangles are perspective from a point `P`: the three lines joining corresponding vertices pass through `P`. The theorem says that the three intersections of corresponding sides form a new line, the Desargues axis.

The diagram uses finite coordinates only. That is a convenience, not a restriction: the incidence tests are homogeneous determinant tests.


In [ ]:
P = hpoint(0.0, 0.0)
A = hpoint(-1.1, 1.25)
B = hpoint(1.35, 0.95)
C = hpoint(-0.25, -1.2)
A1 = hpoint(-1.1 * 1.5, 1.25 * 1.5)
B1 = hpoint(1.35 * 0.5, 0.95 * 0.5)
C1 = hpoint(-0.25 * 0.8, -1.2 * 0.8)

X = meet(join(B, C), join(B1, C1))
Y = meet(join(C, A), join(C1, A1))
Z = meet(join(A, B), join(A1, B1))
axis = join(X, Y)

xlim = (-2.2, 2.2)
ylim = (-2.4, 2.1)
fig, ax = plt.subplots(figsize=(8, 6.2))
for p, q, color, alpha in [
    (A, B, COLORS["blue"], 0.9), (B, C, COLORS["blue"], 0.9), (C, A, COLORS["blue"], 0.9),
    (A1, B1, COLORS["orange"], 0.9), (B1, C1, COLORS["orange"], 0.9), (C1, A1, COLORS["orange"], 0.9),
]:
    pp, qq = point2(p), point2(q)
    ax.plot([pp[0], qq[0]], [pp[1], qq[1]], color=color, lw=2.0, alpha=alpha)
for p, q in [(A, A1), (B, B1), (C, C1)]:
    draw_projective_line(ax, join(p, q), xlim, ylim, color="#8a8f98", lw=1.4, ls="--")
draw_projective_line(ax, axis, xlim, ylim, color=COLORS["red"], lw=2.4)
draw_points(ax, {"P": P}, COLORS["gray"], offset=(0.05, -0.15), size=56)
draw_points(ax, {"A": A, "B": B, "C": C}, COLORS["blue"])
draw_points(ax, {"A1": A1, "B1": B1, "C1": C1}, COLORS["orange"])
draw_points(ax, {"X": X, "Y": Y, "Z": Z}, COLORS["red"], offset=(0.05, 0.07), size=52)
ax.text(-2.05, 1.85, "corresponding vertices meet at P", color=COLORS["gray"])
ax.text(0.45, -2.16, "X, Y, Z share the Desargues axis", color=COLORS["red"], weight="bold")
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_aspect("equal", adjustable="box")
ax.grid(True, color="#e9ecef", lw=0.8)
ax.set_title("Desargues configuration from homogeneous joins and meets", weight="bold")
desargues_path = save_fig(fig, "desargues-perspective-axis.png")

axis_residuals = {label: abs(float(np.dot(point, axis))) for label, point in {"X": X, "Y": Y, "Z": Z}.items()}
perspective_residuals = {
    "PAA1": abs(bracket(P, A, A1)),
    "PBB1": abs(bracket(P, B, B1)),
    "PCC1": abs(bracket(P, C, C1)),
}
CHECKS["desargues_axis_residual"] = max(axis_residuals.values())
CHECKS["desargues_perspective_residual"] = max(perspective_residuals.values())

planes = ["0", "1", "2", "3", "4"]
line_nodes = ["L" + "".join(pair) for pair in combinations(planes, 2)]
point_nodes = ["P" + "".join(triple) for triple in combinations(planes, 3)]
G = nx.Graph()
G.add_nodes_from(line_nodes, part="lines")
G.add_nodes_from(point_nodes, part="points")
for ln in line_nodes:
    pair = set(ln[1:])
    for pn in point_nodes:
        triple = set(pn[1:])
        if pair.issubset(triple):
            G.add_edge(ln, pn)
left_pos = {node: (0, i) for i, node in enumerate(line_nodes)}
right_pos = {node: (3, i + 0.5) for i, node in enumerate(point_nodes)}
pos = {**left_pos, **right_pos}
fig, ax = plt.subplots(figsize=(8.5, 7.2))
nx.draw_networkx_edges(G, pos, ax=ax, width=1.0, alpha=0.45, edge_color="#555555")
nx.draw_networkx_nodes(G, pos, nodelist=line_nodes, node_color=COLORS["blue"], node_size=600, ax=ax, label="lines")
nx.draw_networkx_nodes(G, pos, nodelist=point_nodes, node_color=COLORS["orange"], node_size=600, ax=ax, label="points")
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color="white", font_weight="bold")
ax.text(0, -0.9, "10 pairwise plane intersections: lines", ha="center", color=COLORS["blue"], weight="bold")
ax.text(3, -0.9, "10 triple plane intersections: points", ha="center", color=COLORS["orange"], weight="bold")
ax.set_title("Desargues 10_3 incidence graph from five planes", weight="bold")
ax.set_axis_off()
incidence_graph_path = save_fig(fig, "desargues-10-3-incidence-graph.png")
degrees = dict(G.degree())
CHECKS["desargues_10_3_degrees"] = sorted(set(degrees.values()))

display_artifact(desargues_path, width=720)
display_artifact(incidence_graph_path, width=680)
axis_residuals


## Binomial Proofs: Make the Cancellation Visible

A collinearity such as `A, B, Z` gives a bracket zero. Combining it with the Grassmann-Pluecker relation produces a binomial equality: one product of brackets equals another product of brackets. A binomial proof is a list of such rows whose product cancels almost completely.

The table below records one Desargues proof ledger. The important object is not the visual similarity of the brackets; it is the exponent count of each canonical bracket after all left sides and all right sides are multiplied.


In [ ]:
LABEL_ORDER = ["P", "A", "B", "C", "A1", "B1", "C1", "X", "Y", "Z"]
ORDER = {label: i for i, label in enumerate(LABEL_ORDER)}

binomial_rows = [
    ("C,Y,A collinear", [("C", "Y", "B"), ("C", "A", "X")], [("C", "Y", "X"), ("C", "A", "B")]),
    ("A,B,X collinear", [("A", "B", "C"), ("A", "X", "P")], [("A", "B", "P"), ("A", "X", "C")]),
    ("P,A1,A collinear", [("P", "A1", "X"), ("P", "A", "B")], [("P", "A1", "B"), ("P", "A", "X")]),
    ("P,C,C1 collinear", [("P", "C", "Z"), ("P", "C1", "B1")], [("P", "C", "B1"), ("P", "C1", "Z")]),
    ("C1,Z,B1 collinear", [("C1", "Z", "P"), ("C1", "B1", "A1")], [("C1", "Z", "A1"), ("C1", "B1", "P")]),
    ("A1,Y,C1 collinear", [("A1", "Y", "B1"), ("A1", "C1", "Z")], [("A1", "Y", "Z"), ("A1", "C1", "B1")]),
    ("P,B,B1 collinear", [("P", "B", "A1"), ("P", "B1", "C")], [("P", "B", "C"), ("P", "B1", "A1")]),
    ("C,B,Z collinear", [("C", "B", "P"), ("C", "Z", "Y")], [("C", "B", "Y"), ("C", "Z", "P")]),
    ("A1,B1,X collinear", [("A1", "B1", "P"), ("A1", "X", "Y")], [("A1", "B1", "Y"), ("A1", "X", "P")]),
]


def bracket_label(term):
    return "[" + " ".join(term) + "]"


def parity(term):
    inv = 0
    indices = [ORDER[item] for item in term]
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            inv += indices[i] > indices[j]
    return -1 if inv % 2 else 1


def canonical(term):
    return tuple(sorted(term, key=lambda item: ORDER[item])), parity(term)


left_counts = Counter()
right_counts = Counter()
left_sign = 1
right_sign = 1
rows_for_table = []
for hypothesis, left, right in binomial_rows:
    for term in left:
        can, sign = canonical(term)
        left_counts[can] += 1
        left_sign *= sign
    for term in right:
        can, sign = canonical(term)
        right_counts[can] += 1
        right_sign *= sign
    rows_for_table.append({
        "hypothesis": hypothesis,
        "left product": " ".join(bracket_label(term) for term in left),
        "right product": " ".join(bracket_label(term) for term in right),
    })

left_remainder = list((left_counts - right_counts).elements())
right_remainder = list((right_counts - left_counts).elements())
ledger_rows = []
for term in sorted(set(left_counts) | set(right_counts), key=lambda t: [ORDER[x] for x in t]):
    ledger_rows.append({
        "canonical bracket": bracket_label(term),
        "left exponent": left_counts[term],
        "right exponent": right_counts[term],
        "net left minus right": left_counts[term] - right_counts[term],
    })

binomial_table_path = save_csv(rows_for_table, "desargues-binomial-proof-rows.csv")
ledger_table_path = save_csv(ledger_rows, "desargues-binomial-cancellation-ledger.csv")

fig, ax = plt.subplots(figsize=(12.5, 5.4))
ax.set_axis_off()
summary = [
    ["rows multiplied", len(binomial_rows)],
    ["canonical brackets on left", sum(left_counts.values())],
    ["canonical brackets on right", sum(right_counts.values())],
    ["survive on left", " ".join(bracket_label(t) for t in left_remainder)],
    ["survive on right", " ".join(bracket_label(t) for t in right_remainder)],
    ["next step", "Grassmann-Pluecker turns the survivor equality into the axis condition [X Y Z]=0, assuming the auxiliary bracket is nonzero"],
]
table = ax.table(cellText=summary, colLabels=["ledger item", "value"], loc="center", cellLoc="left", colLoc="left", colWidths=[0.25, 0.75])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.0, 1.6)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#d0d5dd")
    if row == 0:
        cell.set_facecolor(COLORS["gray"])
        cell.get_text().set_color("white")
        cell.get_text().set_weight("bold")
    elif row in (4, 5):
        cell.set_facecolor("#fff4e6")
    else:
        cell.set_facecolor("white")
ax.set_title("Desargues binomial proof as a cancellation ledger", weight="bold", pad=16)
binomial_path = save_fig(fig, "binomial-cancellation-ledger.png")

CHECKS["binomial_left_remainder"] = [bracket_label(t) for t in left_remainder]
CHECKS["binomial_right_remainder"] = [bracket_label(t) for t in right_remainder]
CHECKS["binomial_signs_match"] = left_sign == right_sign
CHECKS["binomial_cancellation_ok"] = (
    sorted(left_remainder) == sorted([("C", "Y", "Z"), ("A1", "X", "Y")])
    and sorted(right_remainder) == sorted([("C", "X", "Y"), ("A1", "Y", "Z")])
    and left_sign == right_sign
)

display_artifact(binomial_path, width=780)
pd.DataFrame(rows_for_table)


## Chains and Cycles of Cross-Ratios

A perspectivity from one line to another is a projective map on that line. In coordinates on the line it is a Mobius transformation. That makes the cross-ratio the correct passenger: it survives every step of the chain.

The cycle check here uses maps that fix one marked point `D`. After several transports, the projectivity that sends the final `A` and `B` back to their starting positions while fixing `D` also sends the final `C` back. That is the computational version of the closing incidence.


In [ ]:
def mobius_apply(M, x):
    vec = np.array([float(x), 1.0])
    y = M @ vec
    return float(y[0] / y[1])


start_values = {"A": 0.35, "B": 1.15, "C": 2.05, "D": 0.0}
chain_maps = [
    np.array([[1.25, 0.0], [0.18, 1.0]]),
    np.array([[0.82, 0.0], [-0.10, 1.0]]),
    np.array([[1.14, 0.0], [0.07, 1.0]]),
    np.array([[0.91, 0.0], [0.12, 1.0]]),
    np.array([[1.08, 0.0], [-0.06, 1.0]]),
]
labels = ["l1"]
tracks = {name: [value] for name, value in start_values.items()}
for i, M in enumerate(chain_maps, start=2):
    labels.append(f"l{i}")
    for name in tracks:
        tracks[name].append(mobius_apply(M, tracks[name][-1]))

cr_values = []
for j in range(len(labels)):
    cr_values.append(float(cross_ratio(tracks["A"][j], tracks["B"][j], tracks["C"][j], tracks["D"][j])))
cr0 = cr_values[0]
max_cr_error = max(abs(value - cr0) for value in cr_values)

A_final, B_final, C_final = tracks["A"][-1], tracks["B"][-1], tracks["C"][-1]
A0, B0, C0 = start_values["A"], start_values["B"], start_values["C"]
linear = np.array([[A_final, -A0 * A_final], [B_final, -B0 * B_final]], dtype=float)
rhs = np.array([A0, B0], dtype=float)
close_a, close_c = np.linalg.solve(linear, rhs)
closing = np.array([[close_a, 0.0], [close_c, 1.0]])
C_closed = mobius_apply(closing, C_final)
cycle_closure_error = abs(C_closed - C0)

fig, ax = plt.subplots(figsize=(10.5, 5.2))
xcoords = np.arange(len(labels))
for j, label in enumerate(labels):
    ax.axvline(j, color="#d7dce2", lw=1.0, zorder=0)
track_colors = {"A": COLORS["blue"], "B": COLORS["orange"], "C": COLORS["green"], "D": COLORS["gray"]}
for name, vals in tracks.items():
    ax.plot(xcoords, vals, marker="o", lw=2.2, color=track_colors[name], label=name)
    for x, y in zip(xcoords, vals):
        ax.text(x + 0.04, y + 0.03, name, color=track_colors[name], fontsize=8, weight="bold")
for j, cr in enumerate(cr_values):
    ax.text(j, max(max(v) for v in tracks.values()) + 0.18, f"CR={cr:.6f}", ha="center", fontsize=8, color="#333333")
ax.set_xticks(xcoords, labels)
ax.set_ylabel("coordinate on current line")
ax.set_title("Cross-ratio transport through a chain of perspectivities", weight="bold")
ax.grid(True, axis="y", color="#edf0f3")
ax.legend(ncol=4, loc="lower right")
cross_ratio_path = save_fig(fig, "cross-ratio-chain-transport.png")

plotly_fig = go.Figure()
for name, vals in tracks.items():
    plotly_fig.add_trace(go.Scatter(
        x=labels,
        y=vals,
        mode="lines+markers+text",
        text=[name] * len(vals),
        textposition="top center",
        name=name,
        hovertemplate="%{text} on %{x}<br>coordinate=%{y:.8f}<extra></extra>",
    ))
plotly_fig.update_layout(
    title="Interactive cross-ratio cycle lab: the value is transported unchanged",
    xaxis_title="line in the chain",
    yaxis_title="line coordinate",
    template="plotly_white",
    height=520,
)
html_path = artifact_path(ARTIFACT_ROOT, "html", "cross-ratio-cycle-lab.html")
plotly_fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
ARTIFACTS.append(html_path)

CHECKS["cross_ratio_error"] = float(max_cr_error)
CHECKS["cross_ratio_cycle_closure_error"] = float(cycle_closure_error)

display_artifact(cross_ratio_path, width=780)
display_artifact(html_path, width="100%", height=460)
{"cross_ratio": cr0, "max_chain_error": max_cr_error, "cycle_closure_error": cycle_closure_error}


## Ceva and Menelaus: The Same Product, Two Signs

Ceva and Menelaus look metric because they use directed segment ratios. The bracket formula shows why the product is projective: the same ratio can be written as a quotient of determinants.

The side-by-side diagram uses the same triangle. On the left, three cevians meet, and the product of directed ratios is `+1`. On the right, a transversal cuts the three sides, and the product is `-1`.


In [ ]:
A2 = np.array([0.0, 0.0])
B2 = np.array([4.0, 0.0])
C2 = np.array([0.7, 3.2])
D2 = np.array([1.6, 1.1])

Xc = intersect_affine(C2, D2, A2, B2)
Yc = intersect_affine(A2, D2, B2, C2)
Zc = intersect_affine(B2, D2, C2, A2)
ceva_ratios = [
    oriented_ratio(A2, Xc, B2),
    oriented_ratio(B2, Yc, C2),
    oriented_ratio(C2, Zc, A2),
]
ceva_product = math.prod(ceva_ratios)

L0 = np.array([-0.4, 1.2])
L1 = np.array([4.2, 2.4])
Xm = intersect_affine(L0, L1, A2, B2)
Ym = intersect_affine(L0, L1, B2, C2)
Zm = intersect_affine(L0, L1, C2, A2)
menelaus_ratios = [
    oriented_ratio(A2, Xm, B2),
    oriented_ratio(B2, Ym, C2),
    oriented_ratio(C2, Zm, A2),
]
menelaus_product = math.prod(menelaus_ratios)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharex=True, sharey=True)
for ax, title in zip(axes, ["Ceva: concurrent cevians", "Menelaus: collinear cuts"]):
    tri = np.array([A2, B2, C2, A2])
    ax.plot(tri[:, 0], tri[:, 1], color="#1f2937", lw=2)
    for label, pt in {"A": A2, "B": B2, "C": C2}.items():
        ax.scatter(pt[0], pt[1], s=44, color="#1f2937", zorder=5)
        ax.text(pt[0] + 0.05, pt[1] + 0.05, label, weight="bold")
    ax.set_title(title, weight="bold")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, color="#eef1f4")
    ax.set_xlim(-5.6, 4.7)
    ax.set_ylim(-0.5, 3.7)

axes[0].scatter(D2[0], D2[1], s=58, color=COLORS["red"], zorder=6)
axes[0].text(D2[0] + 0.05, D2[1] + 0.05, "D", color=COLORS["red"], weight="bold")
for vertex, cut in [(A2, Yc), (B2, Zc), (C2, Xc)]:
    axes[0].plot([vertex[0], cut[0]], [vertex[1], cut[1]], color=COLORS["blue"], lw=1.8)
draw_points(axes[0], {"X": Xc, "Y": Yc, "Z": Zc}, COLORS["blue"], offset=(0.05, 0.05), size=44)
axes[0].text(0.25, -0.38, f"product = {ceva_product:.6f}", color=COLORS["blue"], weight="bold")

axes[1].plot([L0[0], L1[0]], [L0[1], L1[1]], color=COLORS["red"], lw=2.1)
draw_points(axes[1], {"X": Xm, "Y": Ym, "Z": Zm}, COLORS["red"], offset=(0.05, 0.05), size=44)
axes[1].text(-5.1, -0.38, f"product = {menelaus_product:.6f}", color=COLORS["red"], weight="bold")
fig.suptitle("Directed side-ratio products distinguish Ceva from Menelaus", weight="bold")
ceva_menelaus_path = save_fig(fig, "ceva-menelaus-ratio-products.png")

CHECKS["ceva_product"] = float(ceva_product)
CHECKS["ceva_product_error"] = abs(float(ceva_product) - 1.0)
CHECKS["menelaus_product"] = float(menelaus_product)
CHECKS["menelaus_product_error"] = abs(float(menelaus_product) + 1.0)
CHECKS["ceva_denominators_nonzero"] = all(abs(v) > 1e-10 for v in ceva_ratios + menelaus_ratios)

ratio_rows = [
    {"theorem": "Ceva", "factor": "AX/XB", "value": ceva_ratios[0]},
    {"theorem": "Ceva", "factor": "BY/YC", "value": ceva_ratios[1]},
    {"theorem": "Ceva", "factor": "CZ/ZA", "value": ceva_ratios[2]},
    {"theorem": "Menelaus", "factor": "AX/XB", "value": menelaus_ratios[0]},
    {"theorem": "Menelaus", "factor": "BY/YC", "value": menelaus_ratios[1]},
    {"theorem": "Menelaus", "factor": "CZ/ZA", "value": menelaus_ratios[2]},
]
for row in ratio_rows:
    row["inspection target"] = "track this directed side ratio; the product is +1 for concurrence and -1 for a transversal"
    row["validation role"] = "orientation-sensitive factor in the Chapter 15 Ceva/Menelaus product"
ratio_table_path = save_csv(ratio_rows, "ceva-menelaus-ratio-products.csv")

display_artifact(ceva_menelaus_path, width=780)
pd.DataFrame(ratio_rows)


## Glued Configurations: Interior Edge Factors Disappear

The chapter's gluing idea is a proof by accounting. Put a Ceva or Menelaus expression on each oriented triangle. When two faces share an edge with opposite orientations, the quotient from one face is the reciprocal of the quotient from the other. Multiplying face equations cancels the shared data.

The disk example below shows one shared edge cancellation. The tetrahedron ledger then checks the closed 2-cycle case: every edge appears once in each orientation, so every exponent is zero.


In [ ]:
A3 = np.array([0.0, 0.0])
B3 = np.array([3.0, 0.0])
C3 = np.array([1.0, 2.35])
D3 = np.array([4.2, 2.0])
shared = 0.56 * B3 + 0.44 * C3
fig, ax = plt.subplots(figsize=(8.2, 5.2))
for tri, color in [([A3, B3, C3, A3], COLORS["blue"]), ([C3, B3, D3, C3], COLORS["orange"])]:
    arr = np.array(tri)
    ax.plot(arr[:, 0], arr[:, 1], color=color, lw=2.3)
for label, pt in {"A": A3, "B": B3, "C": C3, "D": D3}.items():
    ax.scatter(pt[0], pt[1], s=45, color="#20242a", zorder=4)
    ax.text(pt[0] + 0.05, pt[1] + 0.05, label, weight="bold")
ax.plot([B3[0], C3[0]], [B3[1], C3[1]], color=COLORS["red"], lw=4.0, alpha=0.45)
ax.scatter(shared[0], shared[1], s=64, color=COLORS["red"], zorder=6, edgecolor="white")
ax.text(shared[0] + 0.05, shared[1] + 0.05, "shared edge point", color=COLORS["red"], weight="bold")
ax.text(0.32, 1.05, "face ABC uses B -> C", color=COLORS["blue"], weight="bold")
ax.text(2.18, 1.78, "face CBD uses C -> B", color=COLORS["orange"], weight="bold")
ax.annotate("same edge, reciprocal quotient", xy=shared, xytext=(1.45, 2.72), arrowprops={"arrowstyle": "->", "color": COLORS["red"], "lw": 1.6}, color=COLORS["red"], weight="bold")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-0.45, 4.7)
ax.set_ylim(-0.35, 3.1)
ax.grid(True, color="#edf0f3")
ax.set_title("Gluing two Ceva faces: the interior edge quotient cancels", weight="bold")
glued_path = save_fig(fig, "glued-ceva-cancellation-disk.png")


def edge_exponents(faces):
    counts = Counter()
    rows = []
    for face_name, tri in faces:
        oriented_edges = [(tri[0], tri[1]), (tri[1], tri[2]), (tri[2], tri[0])]
        for edge in oriented_edges:
            key = tuple(sorted(edge))
            sign = 1 if edge == key else -1
            counts[key] += sign
            rows.append({"face": face_name, "oriented edge": f"{edge[0]}->{edge[1]}", "canonical edge": "-".join(key), "exponent contribution": sign, "inspection target": "opposite orientations on a shared edge contribute reciprocal factors", "validation role": "sum canonical edge exponents; interior or closed-cycle edges should cancel"})
    return counts, rows


disk_counts, disk_rows = edge_exponents([("ABC", ("A", "B", "C")), ("CBD", ("C", "B", "D"))])
tetra_faces = [("ABC", ("A", "B", "C")), ("ADB", ("A", "D", "B")), ("ACD", ("A", "C", "D")), ("BDC", ("B", "D", "C"))]
tetra_counts, tetra_rows = edge_exponents(tetra_faces)
for row in tetra_rows:
    row["closed-cycle total exponent"] = tetra_counts[tuple(row["canonical edge"].split("-"))]

save_csv(disk_rows, "glued-disk-edge-cancellation.csv")
tetra_table_path = save_csv(tetra_rows, "tetrahedron-cycle-edge-cancellation.csv")
CHECKS["disk_shared_edge_cancels"] = disk_counts[("B", "C")] == 0
CHECKS["tetrahedron_cycle_exponents"] = {"-".join(edge): int(value) for edge, value in sorted(tetra_counts.items())}
CHECKS["tetrahedron_cycle_cancels"] = all(value == 0 for value in tetra_counts.values())
CHECKS["four_menelaus_faces_even"] = (4 % 2 == 0)

display_artifact(glued_path, width=720)
pd.DataFrame(tetra_rows)


## Bracket Expressions and Exact Theorem Checks

The figures above are useful only if their algebraic claims are checked. This cell uses exact rational arithmetic for the Grassmann-Pluecker relation and for a rational Desargues instance. It also records a numeric projective-invariance lab for the Ceva product.


In [ ]:
def smat(values):
    return sp.Matrix(values)


def scross(u, v):
    return sp.Matrix(u).cross(sp.Matrix(v))


def sbracket(a, b, c):
    return sp.Matrix.hstack(sp.Matrix(a), sp.Matrix(b), sp.Matrix(c)).det()


ZA = smat([sp.Rational(1), sp.Rational(2), sp.Rational(1)])
AA = smat([sp.Rational(3), sp.Rational(-1), sp.Rational(1)])
BB = smat([sp.Rational(-2), sp.Rational(1), sp.Rational(1)])
QQ = smat([sp.Rational(2), sp.Rational(5), sp.Rational(1)])
RR = smat([sp.Rational(-1), sp.Rational(4), sp.Rational(1)])
gp_value = sp.expand(
    sbracket(ZA, AA, BB) * sbracket(ZA, QQ, RR)
    - sbracket(ZA, AA, QQ) * sbracket(ZA, BB, RR)
    + sbracket(ZA, AA, RR) * sbracket(ZA, BB, QQ)
)

Ps = smat([0, 0, 1])
As = smat([sp.Rational(-3, 2), sp.Rational(5, 4), 1])
Bs = smat([sp.Rational(7, 5), sp.Rational(9, 10), 1])
Cs = smat([sp.Rational(-1, 3), sp.Rational(-6, 5), 1])
A1s = smat([sp.Rational(3, 2) * As[0], sp.Rational(3, 2) * As[1], 1])
B1s = smat([sp.Rational(1, 2) * Bs[0], sp.Rational(1, 2) * Bs[1], 1])
C1s = smat([sp.Rational(4, 5) * Cs[0], sp.Rational(4, 5) * Cs[1], 1])
Xs = scross(scross(Bs, Cs), scross(B1s, C1s))
Ys = scross(scross(Cs, As), scross(C1s, A1s))
Zs = scross(scross(As, Bs), scross(A1s, B1s))
exact_desargues_axis = sp.simplify(sbracket(Xs, Ys, Zs))

A_h, B_h, C_h = hp(A2), hp(B2), hp(C2)
X_h, Y_h, Z_h = hp(Xc), hp(Yc), hp(Zc)
P_h = hpoint(2.2, -1.1)
H = np.array([[1.1, 0.18, 0.4], [-0.16, 0.93, 0.25], [0.05, -0.04, 1.0]])
original_bracket_product = bracket_product(P_h, A_h, B_h, C_h, X_h, Y_h, Z_h)
transformed_bracket_product = bracket_product(*(H @ p for p in [P_h, A_h, B_h, C_h, X_h, Y_h, Z_h]))
projective_invariance_error = abs(original_bracket_product - transformed_bracket_product)

symbolic_rows = [
    {"check": "Grassmann-Pluecker determinant identity", "result": str(gp_value), "passes": gp_value == 0},
    {"check": "Exact rational Desargues bracket [X Y Z]", "result": str(exact_desargues_axis), "passes": exact_desargues_axis == 0},
    {"check": "Ceva bracket product before homography", "result": f"{original_bracket_product:.12g}", "passes": abs(original_bracket_product - 1.0) < 1e-9},
    {"check": "Ceva bracket product after homography", "result": f"{transformed_bracket_product:.12g}", "passes": projective_invariance_error < 1e-9},
]
for row in symbolic_rows:
    row["inspection target"] = "connect a visible configuration claim to an exact determinant or invariant calculation"
    row["validation role"] = "fails when the bracket formula, incidence construction, or projective-invariance convention is wrong"
symbolic_table_path = save_csv(symbolic_rows, "symbolic-and-numeric-theorem-checks.csv")
CHECKS["grassmann_pluecker_exact_zero"] = bool(gp_value == 0)
CHECKS["desargues_exact_bracket_zero"] = bool(exact_desargues_axis == 0)
CHECKS["projective_invariance_error"] = float(projective_invariance_error)
CHECKS["symbolic_checks_pass"] = all(row["passes"] for row in symbolic_rows)

pd.DataFrame(symbolic_rows)


## Artifact Gallery

These direct links make the notebook readable in a static viewer after execution.

![Proof-machine route map](../../artifacts/chapter-15-configurations-theorems-and-bracket-expressions/figures/proof-machines-route-map.png)

![Desargues perspective axis](../../artifacts/chapter-15-configurations-theorems-and-bracket-expressions/figures/desargues-perspective-axis.png)

![Cross-ratio chain transport](../../artifacts/chapter-15-configurations-theorems-and-bracket-expressions/figures/cross-ratio-chain-transport.png)

![Ceva and Menelaus ratio products](../../artifacts/chapter-15-configurations-theorems-and-bracket-expressions/figures/ceva-menelaus-ratio-products.png)

![Glued Ceva cancellation disk](../../artifacts/chapter-15-configurations-theorems-and-bracket-expressions/figures/glued-ceva-cancellation-disk.png)


## Applied Lab: Perturb, Then Recheck

The smallest useful experiment is to move the ingredients and ask which number should stay fixed. Here the Ceva point moves inside the same triangle. The cut points move, but the product remains `+1` as long as no denominator degenerates. The table is not a proof; it is a diagnostic that catches wrong orientation conventions immediately.


In [ ]:
lab_points = [np.array([1.1, 0.85]), np.array([1.6, 1.1]), np.array([2.2, 1.55]), np.array([0.9, 1.35])]
lab_rows = []
for index, D_lab in enumerate(lab_points, start=1):
    X_lab = intersect_affine(C2, D_lab, A2, B2)
    Y_lab = intersect_affine(A2, D_lab, B2, C2)
    Z_lab = intersect_affine(B2, D_lab, C2, A2)
    factors = [
        oriented_ratio(A2, X_lab, B2),
        oriented_ratio(B2, Y_lab, C2),
        oriented_ratio(C2, Z_lab, A2),
    ]
    product = math.prod(factors)
    lab_rows.append({
        "trial": index,
        "ceva point x": D_lab[0],
        "ceva point y": D_lab[1],
        "AX/XB": factors[0],
        "BY/YC": factors[1],
        "CZ/ZA": factors[2],
        "product": product,
        "absolute error from 1": abs(product - 1.0),
    })
lab_table_path = save_csv(lab_rows, "ceva-perturbation-lab.csv")
CHECKS["ceva_lab_max_error"] = max(row["absolute error from 1"] for row in lab_rows)
pd.DataFrame(lab_rows)


## Takeaways

- Desargues's theorem can be inspected as an incidence picture and as a symmetric `10_3` configuration.
- A binomial proof is a cancellation ledger for bracket monomials; the surviving equality is designed to trigger a Grassmann-Pluecker consequence.
- Cross-ratio chains and cycles are one-dimensional projective proof machines: every perspectivity transports the same invariant.
- Ceva and Menelaus use the same directed ratio product, with `+1` for concurrence and `-1` for a transversal.
- Gluing Ceva/Menelaus faces is a topological accounting argument: interior edge factors disappear, and a final boundary or face condition remains.


In [ ]:
all_paths = ARTIFACTS + TABLES + [storyboard_path]
assert_artifacts(all_paths, min_size=256)

assert CHECKS["storyboard_visual_count"] >= 8
assert CHECKS["desargues_axis_residual"] < 1e-9
assert CHECKS["desargues_perspective_residual"] < 1e-9
assert CHECKS["desargues_10_3_degrees"] == [3]
assert CHECKS["binomial_cancellation_ok"]
assert CHECKS["cross_ratio_error"] < 1e-9
assert CHECKS["cross_ratio_cycle_closure_error"] < 1e-9
assert CHECKS["ceva_product_error"] < 1e-9
assert CHECKS["menelaus_product_error"] < 1e-9
assert CHECKS["disk_shared_edge_cancels"]
assert CHECKS["tetrahedron_cycle_cancels"]
assert CHECKS["four_menelaus_faces_even"]
assert CHECKS["symbolic_checks_pass"]
assert CHECKS["projective_invariance_error"] < 1e-9
assert CHECKS["ceva_lab_max_error"] < 1e-9

image_summaries = []
for path in ARTIFACTS:
    if path.suffix.lower() == ".png":
        item = image_stats(path)
        item["path"] = rel(path)
        image_summaries.append(item)
        assert item["pixel_std"] > 1.0

CHECKS["all_files_exist"] = all(path.exists() for path in all_paths)
CHECKS["artifacts"] = [rel(path) for path in all_paths]
CHECKS["image_stats"] = image_summaries
CHECKS["numeric_checks"] = {
    "desargues_axis_residual": CHECKS["desargues_axis_residual"],
    "cross_ratio_error": CHECKS["cross_ratio_error"],
    "cross_ratio_cycle_closure_error": CHECKS["cross_ratio_cycle_closure_error"],
    "ceva_product_error": CHECKS["ceva_product_error"],
    "menelaus_product_error": CHECKS["menelaus_product_error"],
    "projective_invariance_error": CHECKS["projective_invariance_error"],
}

visual_checks_path = save_json(CHECKS, ARTIFACT_ROOT, "checks", "visual-checks.json")
final_sanity = {
    "chapter": 15,
    "source_span": "printed pp. 269-294 / PDF pp. 291-316",
    "checks_passed": True,
    "core_identities": {
        "Desargues axis residual < 1e-9": CHECKS["desargues_axis_residual"],
        "Grassmann-Pluecker exact zero": CHECKS["grassmann_pluecker_exact_zero"],
        "Exact rational Desargues bracket zero": CHECKS["desargues_exact_bracket_zero"],
        "Ceva product error": CHECKS["ceva_product_error"],
        "Menelaus product error": CHECKS["menelaus_product_error"],
        "Cross-ratio chain error": CHECKS["cross_ratio_error"],
        "Tetrahedron cycle cancels": CHECKS["tetrahedron_cycle_cancels"],
    },
    "artifact_count": len(all_paths) + 2,
    "artifacts": [rel(path) for path in all_paths] + [rel(visual_checks_path), rel(ARTIFACT_ROOT / "checks" / "final-sanity.json")],
}
final_sanity_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
assert_artifacts([visual_checks_path, final_sanity_path], min_size=256)
final_sanity
